# Lifecycle anomaly detection and utility experiment

Orchestrates analysis from `state extractor` and `utility part experiment` via the `functions/` package. Algorithms are unchanged; outputs go to `results/`.


## 1. Load DuckDB and build weekly counts


In [ ]:
from pathlib import Path

from functions.config import (
    COMMITIZEN_DUCKDB,
    OCEL2_SQLITE,
    RESULTS_FIGURES,
    RESULTS_TABLES,
)
from functions.data_loading import (
    build_weekly_issue_counts,
    load_events_per_obj,
    load_objects_attributes,
)
from functions.plotting_anomaly import plot_weekly_event_counts

RESULTS_TABLES.mkdir(parents=True, exist_ok=True)
RESULTS_FIGURES.mkdir(parents=True, exist_ok=True)

print("DuckDB:", COMMITIZEN_DUCKDB)
print("OCEL2:", OCEL2_SQLITE)

eventsPerobj_df = load_events_per_obj()
objects_attributes = load_objects_attributes()
weekly = build_weekly_issue_counts(eventsPerobj_df)
plot_weekly_event_counts(weekly)
weekly.head()


## 2. Rolling IQR + lexicon state extraction


In [ ]:
from functions.anomaly_pipeline import run_anomaly_state_extraction

weekly_105_anomaly_frame, anomaly_context, anomaly_states, event_105_title_join = (
    run_anomaly_state_extraction(weekly, eventsPerobj_df, objects_attributes)
)
anomaly_states.head()


## 3. Export anomaly object / event IDs


In [ ]:
from functions.anomaly_ids import export_anomaly_ids

anomaly_object_ids, anomaly_event_ids = export_anomaly_ids(anomaly_context)
anomaly_object_ids.head()


## 4. Anomaly plots


In [ ]:
from functions.plotting_anomaly import plot_anomaly_figures

plot_anomaly_figures(weekly_105_anomaly_frame, anomaly_states)


## 5. Commit context and type classification


In [ ]:
from functions.commits import build_commit_context, build_commit_typeclass
from functions.plotting_anomaly import plot_commit_category_stack

commit_context, event_43_message_join = build_commit_context(
    weekly_105_anomaly_frame, eventsPerobj_df, objects_attributes
)
commit_typeclass_per_week = build_commit_typeclass(commit_context)
plot_commit_category_stack(commit_context, commit_typeclass_per_week)


## 6. Flatten OCEL2 issue log


In [ ]:
from functions.ocel_flatten import flatten_ocel2_issue_log

flat = flatten_ocel2_issue_log()


## 7. Vitalizing subset from anomaly object IDs


In [ ]:
from functions.ocel_flatten import build_vitalizing_subset

anomaly_events_df, anomaly_df = build_vitalizing_subset(flat, anomaly_object_ids)


## 8. Preprocess + random control (MAX_REP / SEED unchanged)


In [ ]:
from functions.preprocessing import build_preprocessed_logs

prep = build_preprocessed_logs(flat, anomaly_events_df)
flat_df_clean = prep["flat_df_clean"]
vitalizing_df_clean = prep["vitalizing_df_clean"]
random_case_control_df_clean = prep["random_case_control_df_clean"]
clean_log_groups_df = prep["clean_log_groups_df"]
all_cases_clean = prep["all_cases_clean"]
num_vital_cases_clean = prep["num_vital_cases_clean"]


## 9. Discovery F1 evaluation on clean logs


In [ ]:
from functions.discovery import evaluate_clean_logs

consolidated_results_table_clean, clean_log_groups_eventlog, miners = evaluate_clean_logs(
    clean_log_groups_df
)
consolidated_results_table_clean.head()


## 10. Utility figures


In [ ]:
from functions.plotting_utility import plot_utility_figures

plot_utility_figures(consolidated_results_table_clean)


## 11. Log complexity metrics (process-complexity / EPA)


In [ ]:
from functions.complexity_metrics import compute_log_complexity_table

log_complexity_similarity_table = compute_log_complexity_table(clean_log_groups_eventlog)
log_complexity_similarity_table


## 12. Seed reproducibility


In [ ]:
from functions.plotting_utility import plot_repro_slopegraph
from functions.reproducibility import run_seed_reproducibility

repro = run_seed_reproducibility(
    flat_df_clean=flat_df_clean,
    all_cases_clean=all_cases_clean,
    num_vital_cases_clean=num_vital_cases_clean,
    miners=miners,
    consolidated_results_table_clean=consolidated_results_table_clean,
)
consolidated_results_table_clean_repro = repro["consolidated_results_table_clean_repro"]
plot_repro_slopegraph(consolidated_results_table_clean_repro)
